# 03 · Data dictionary

**Phase 1 deliverable: a full data dictionary**, generated **from the database itself**, so types, row counts, % missing and distinct counts are measured, not typed by hand.

* **Raw tables** (13): every column with its type, % missing, number of distinct values, an example and a plain-English description.
* **Clean, feature and quality objects** (`clean`, `feat`, `dq`): what each table or view holds, which SQL file builds it, and the columns the clean layer adds.
* **KPI definitions**, from `clean.kpi_definitions`.

Descriptions live in `notebooks/dictionary_text.py`: edit the wording there and re-run.
**Outputs** (folder `reports/03_data_dictionary/`): `data_dictionary.md`, `data_dictionary.xlsx` and `Veltrix_data_dictionary.docx`.

The two largest tables (web_sessions, property_views) use PostgreSQL's own column statistics (`pg_stats`) for % missing and distinct counts; they are marked *approx.*

In [1]:
# First time only:  %pip install sqlalchemy psycopg2-binary python-docx openpyxl

In [2]:
import re
import pandas as pd
from IPython.display import display, Markdown
import vx
from dictionary_text import TABLE_DESC, DESC, CLEAN_EXTRA, OBJ_DESC

vx.connect()
OUT = vx.out_dir("03_data_dictionary")
Q = vx.Q

Connected to database 'Veltrix' on localhost:5432 as postgres


## 1. Raw tables: measured from the data

In [3]:
BIG = 1_000_000
rows_out, tables_out = [], []
for t, tdesc in TABLE_DESC.items():
    n = vx.V(f"SELECT count(*) FROM raw.{t}")
    cols = Q("""SELECT column_name, data_type FROM information_schema.columns
                WHERE table_schema = 'raw' AND table_name = :t ORDER BY ordinal_position""", t=t)
    if n > BIG:     # big tables: PostgreSQL statistics (fast); refresh them first
        vx.run(f"ANALYZE raw.{t}")
        st = Q("SELECT attname, null_frac, n_distinct FROM pg_stats WHERE schemaname = 'raw' AND tablename = :t", t=t).set_index("attname")
        nulls = {c: 100 * float(st.null_frac.get(c, 0)) for c in cols.column_name}
        dist = {c: (int(-st.n_distinct[c] * n) if st.n_distinct.get(c, 0) < 0 else int(st.n_distinct.get(c, 0))) for c in cols.column_name}
    else:           # exact counts in one query per table
        parts = ", ".join(f'count("{c}") AS "n_{c}", count(DISTINCT "{c}") AS "d_{c}"' for c in cols.column_name)
        res = Q(f"SELECT {parts} FROM raw.{t}").iloc[0]
        nulls = {c: 100 * (1 - res[f"n_{c}"] / n) if n else 0 for c in cols.column_name}
        dist = {c: int(res[f"d_{c}"]) for c in cols.column_name}
    ex = Q(f"SELECT * FROM raw.{t} LIMIT 200")
    for c, typ in cols.itertuples(index=False):
        v = ex[c].dropna()
        e = str(v.iloc[0]) if len(v) else ""
        rows_out.append(dict(schema="raw", table=t, column=c, type=typ, null_pct=round(nulls[c], 1), distinct=dist[c],
                             approx=n > BIG, example=e[:22] + ("…" if len(e) > 22 else ""), description=DESC.get(c, "")))
    tables_out.append(dict(schema="raw", object=t, kind="table", rows=n, built_by="week1/1_create_tables_and_load.sql", description=tdesc))
    print(f"raw.{t:<20} {n:>12,} rows  {len(cols)} columns")
raw_cols = pd.DataFrame(rows_out)
missing_desc = raw_cols[raw_cols.description == ""]
display(Markdown(f"**{len(raw_cols)} columns** in **{len(TABLE_DESC)} raw tables**; "
                 f"{len(missing_desc)} have no description yet" + (f": {', '.join(missing_desc.column.head(10))}" if len(missing_desc) else ".")))

raw.locations                      36 rows  9 columns
raw.agents                        300 rows  6 columns
raw.customers                 102,000 rows  19 columns
raw.campaigns                     270 rows  10 columns
raw.leads                      95,804 rows  11 columns
raw.interactions              532,789 rows  8 columns
raw.properties                 16,667 rows  19 columns
raw.site_visits               233,593 rows  6 columns
raw.transactions               16,665 rows  14 columns
raw.payments                  106,199 rows  10 columns
raw.campaign_responses        808,953 rows  5 columns
raw.web_sessions            2,003,460 rows  8 columns
raw.property_views         15,163,137 rows  5 columns


**130 columns** in **13 raw tables**; 0 have no description yet.

## 2. Clean, feature and quality objects: what each holds and which SQL file builds it

In [4]:
# Find the SQL file that creates each object by reading the week folders
built_by = {}
for f in sorted(vx.ROOT.glob("week*/*.sql")):
    for m in re.finditer(r"CREATE\s+(?:OR\s+REPLACE\s+)?(?:MATERIALIZED\s+)?(?:TABLE|VIEW)\s+(?:IF\s+NOT\s+EXISTS\s+)?([a-z_]+\.[a-z_0-9]+)",
                         f.read_text(encoding="utf-8", errors="ignore"), flags=re.I):
        built_by.setdefault(m.group(1).lower(), f"{f.parent.name}/{f.name}")
built_by["feat.customer_features_model"] = "notebooks/04_feature_engineering.ipynb"

objs = Q("""SELECT table_schema AS schema, table_name AS object,
                   CASE table_type WHEN 'BASE TABLE' THEN 'table' ELSE 'view' END AS kind
            FROM information_schema.tables WHERE table_schema IN ('clean', 'feat', 'dq')
            UNION ALL SELECT schemaname, matviewname, 'materialized view' FROM pg_matviews WHERE schemaname IN ('clean', 'feat', 'dq')
            ORDER BY 1, 2""")
est = Q("""SELECT n.nspname AS schema, c.relname AS object, c.reltuples::bigint AS est_rows
           FROM pg_class c JOIN pg_namespace n ON n.oid = c.relnamespace WHERE n.nspname IN ('clean', 'feat', 'dq')""")
objs = objs.merge(est, on=["schema", "object"], how="left")
objs["rows"] = objs.apply(lambda r: vx.V(f"SELECT count(*) FROM {r.schema}.{r.object}") if r.kind != "view" else None, axis=1)
objs["built_by"] = [built_by.get(f"{s}.{o}", "") for s, o in zip(objs.schema, objs.object)]
objs["description"] = [OBJ_DESC.get(f"{s}.{o}", "Clean copy of raw." + o.replace("clean_", "") if o.startswith("clean_") else OBJ_DESC.get(f"{s}.{o}", ""))
                       for s, o in zip(objs.schema, objs.object)]
objs = objs[["schema", "object", "kind", "rows", "built_by", "description"]]
display(objs.groupby(["schema", "kind"]).size().unstack(fill_value=0))
objs.head(20)

kind,materialized view,table,view
schema,,,
clean,0,33,37
dq,0,3,1
feat,2,3,5


,schema,object,kind,rows,built_by,description
0,clean,audience_email,view,NaN,week3/13_crm_activation_and_test.sql,Consenting customers per segment for e-mail
1,clean,audience_google,view,NaN,week3/13_crm_activation_and_test.sql,Hashed e-mail per segment for Google Customer ...
2,clean,audience_meta,view,NaN,week3/13_crm_activation_and_test.sql,Hashed e-mail/phone per segment for Meta
3,clean,audience_reach,view,NaN,week3/13_crm_activation_and_test.sql,Reachable customers per segment and channel
4,clean,audience_tiktok,view,NaN,week3/13_crm_activation_and_test.sql,Hashed e-mail/phone per segment for TikTok
5,clean,audience_wechat,view,NaN,week3/13_crm_activation_and_test.sql,"Hashed phone per segment for WeChat (CN, HK, S..."
6,clean,budget_fixes,table,236.0,week1/3_cleaning_and_baselines.sql,"One row per corrected budget: raw value, refer..."
7,clean,calendar,table,2192.0,week2/7_slicers_and_headlines.sql,Date table for Power BI
8,clean,campaign_plan,table,6.0,week3/13_crm_activation_and_test.sql,Next month's campaigns with target_segment
9,clean,chi2_critical_001,table,12.0,week2/9_validation_and_swap.sql,Chi-square critical values at 0.1% significanc...


In [5]:
obj_cols = Q("""SELECT table_schema AS schema, table_name AS object, column_name AS column, data_type AS type
                FROM information_schema.columns WHERE table_schema IN ('clean', 'feat', 'dq')
                UNION ALL
                SELECT n.nspname, c.relname, a.attname, format_type(a.atttypid, a.atttypmod)
                FROM pg_attribute a JOIN pg_class c ON c.oid = a.attrelid JOIN pg_namespace n ON n.oid = c.relnamespace
                WHERE c.relkind = 'm' AND n.nspname IN ('clean', 'feat', 'dq') AND a.attnum > 0 AND NOT a.attisdropped
                ORDER BY 1, 2""")
obj_cols["description"] = [CLEAN_EXTRA.get(o, {}).get(c, DESC.get(c, "")) for o, c in zip(obj_cols.object, obj_cols.column)]
kpis = Q("SELECT * FROM clean.kpi_definitions") if vx.exists("clean.kpi_definitions") else pd.DataFrame()
print(len(objs), "objects,", len(obj_cols), "columns;", len(kpis), "KPI definitions")

84 objects, 707 columns; 9 KPI definitions


## 3. Save: Markdown, Excel and Word

In [6]:
def md_table(df):
    s = "| " + " | ".join(df.columns) + " |\n|" + "---|" * len(df.columns) + "\n"
    for row in df.astype(str).itertuples(index=False):
        s += "| " + " | ".join(x.replace("|", "/").replace("nan", "").replace("None", "") for x in row) + " |\n"
    return s

md = ["# Veltrix data dictionary", "",
      "Generated by `notebooks/03_data_dictionary.ipynb` from the PostgreSQL database (data to 30 Sep 2026). "
      "% missing and distinct counts are measured on the data (*approx.* = PostgreSQL statistics on the two largest tables).", "",
      "## Conventions", "- Money in GBP; timestamps in UK local time; keys are text with a `VLX-` prefix.",
      "- Raw tables are never edited. Every fix lives in a mapping table or in the `clean` schema, so it can be audited and re-run.",
      "- Join path: customers → leads → (interactions, site_visits, transactions → payments); customers → web_sessions → property_views → properties → locations.", ""]
if len(kpis):
    md += ["## KPI definitions (use these everywhere)", "", md_table(kpis)]
for t in TABLE_DESC:
    tc = raw_cols[raw_cols.table == t]
    n = next(x["rows"] for x in tables_out if x["object"] == t)
    md += [f"## raw.{t}", f"{TABLE_DESC[t]} Rows: **{n:,}**", "",
           md_table(tc.assign(distinct=[f"{d:,}" + (" (approx.)" if a else "") for d, a in zip(tc.distinct, tc.approx)])
                    [["column", "type", "null_pct", "distinct", "example", "description"]].rename(columns={"null_pct": "% missing"}))]
md += ["## Clean, feature and quality objects", "", md_table(objs.fillna(""))]
md += ["## Columns the clean layer adds", ""]
for o, cols in CLEAN_EXTRA.items():
    md += [f"### clean.{o}", "", md_table(pd.DataFrame(list(cols.items()), columns=["column", "description"]))]
(OUT / "data_dictionary.md").write_text("\n".join(md), encoding="utf-8")

with pd.ExcelWriter(OUT / "data_dictionary.xlsx", engine="openpyxl") as xw:
    pd.DataFrame(tables_out).to_excel(xw, "raw_tables", index=False)
    raw_cols.to_excel(xw, "raw_columns", index=False)
    objs.to_excel(xw, "clean_feat_dq_objects", index=False)
    obj_cols.to_excel(xw, "clean_feat_dq_columns", index=False)
    if len(kpis):
        kpis.to_excel(xw, "kpi_definitions", index=False)
    for ws in xw.sheets.values():
        ws.freeze_panes = "A2"
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = min(60, max(10, max(len(str(c.value or "")) for c in col[:200]) + 2))
print("Saved data_dictionary.md and data_dictionary.xlsx")

Saved data_dictionary.md and data_dictionary.xlsx


C:\Users\ntjam\AppData\Local\Temp\ipykernel_6704\3821525113.py:28: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  pd.DataFrame(tables_out).to_excel(xw, "raw_tables", index=False)
C:\Users\ntjam\AppData\Local\Temp\ipykernel_6704\3821525113.py:29: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  raw_cols.to_excel(xw, "raw_columns", index=False)
C:\Users\ntjam\AppData\Local\Temp\ipykernel_6704\3821525113.py:30: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  objs.to_excel(xw, "clean_feat_dq_objects", index=False)
C:\Users\ntjam\AppData\Local\Temp\ipykernel_6704\3821525113.py:31: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  obj_cols.to_e

In [7]:
r = vx.Report("Data dictionary", "Phase 1 · Documented analytical database",
              "PostgreSQL database veltrix · data to 30 Sep 2026 · generated by notebooks/03_data_dictionary.ipynb")
r.h1("How to read this dictionary")
r.bullets(["**raw** holds the 13 source tables exactly as loaded. They are never edited.",
           "**clean** holds the cleaned tables and views (clean_*), the mapping tables behind every fix, and everything the dashboard, CRM and campaign test read.",
           "**feat** holds the customer features (behaviour only) and outcomes, kept apart so outcomes never leak into the model.",
           "**dq** holds the data quality log and its helper tables.",
           "Money is in GBP, times are UK local time, keys are text with a VLX- prefix. % missing and distinct counts are measured; "
           "'approx.' marks the two largest tables, where PostgreSQL's statistics are used."])
r.kpis([(f"{len(TABLE_DESC)}", "raw tables"), (f"{sum(x['rows'] for x in tables_out) / 1e6:.1f}M", "raw rows"),
        (f"{len(raw_cols)}", "raw columns"), (f"{len(objs)}", "clean / feat / dq objects")])
if len(kpis):
    r.h1("KPI definitions")
    r.table(kpis, widths_cm=[4, 9, 4])
r.h1("Raw tables")
r.table(pd.DataFrame(tables_out)[["object", "rows", "description"]].rename(columns={"object": "Table", "rows": "Rows", "description": "Description"}),
        widths_cm=[3.5, 2.5, 11])
for t in TABLE_DESC:
    tc = raw_cols[raw_cols.table == t]
    n = next(x["rows"] for x in tables_out if x["object"] == t)
    r.h2(f"raw.{t}  ({n:,} rows)")
    r.p(TABLE_DESC[t], muted=True)
    show = tc.assign(distinct=[f"{d:,}" + (" ~" if a else "") for d, a in zip(tc.distinct, tc.approx)],
                     null_pct=[f"{x:.1f}" for x in tc.null_pct])[["column", "type", "null_pct", "distinct", "example", "description"]]
    show.columns = ["Column", "Type", "% missing", "Distinct", "Example", "Description"]
    r.table(show, font=7.5, widths_cm=[3.4, 2.4, 1.5, 1.7, 2.8, 5.2])
r.page_break()
r.h1("Clean, feature and quality objects")
r.p("Which SQL file builds each object, and what it holds. Views store the query, not the data, so they always reflect the clean tables.")
for sch in ["clean", "feat", "dq"]:
    r.h2(f"Schema {sch}")
    so = objs[objs.schema == sch].copy()
    so["rows"] = so.rows.map(lambda x: "" if pd.isna(x) else f"{int(x):,}")
    so = so[["object", "kind", "rows", "built_by", "description"]]
    so.columns = ["Object", "Kind", "Rows", "Built by", "Description"]
    r.table(so, font=7.5, widths_cm=[3.8, 1.8, 1.6, 4.4, 5.4])
r.h1("Columns the clean layer adds")
for o, cols in CLEAN_EXTRA.items():
    r.h3(f"clean.{o}")
    r.table(pd.DataFrame(list(cols.items()), columns=["Column", "Description"]), font=8, widths_cm=[4.5, 12.5])
r.save(OUT / "Veltrix_data_dictionary.docx")

Saved reports\03_data_dictionary\Veltrix_data_dictionary.docx
